# 🧠 Keras Deep Learning Curriculum
## Complete Beginner-to-Advanced Jupyter Notebook

> **Estimated Total Time:** 60–80 hours  
> **Framework:** Keras (TensorFlow backend)  
> **Level:** Beginner → Advanced → Production  

---

## 📋 Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Keras Core Concepts](#2-keras-core-concepts)
3. [Sequential API — Your First Neural Network](#3-sequential-api)
4. [Functional API — Multi-Input & Multi-Output Models](#4-functional-api)
5. [Model Subclassing — Full Custom Control](#5-model-subclassing)
6. [Custom Layers, Losses & Metrics](#6-custom-layers-losses-metrics)
7. [Callbacks & Training Control](#7-callbacks)
8. [Data Pipelines with tf.data](#8-data-pipelines)
9. [Computer Vision — CNNs & Transfer Learning](#9-computer-vision)
10. [Natural Language Processing — Text & Transformers](#10-nlp)
11. [Time Series Forecasting](#11-time-series)
12. [Generative Models — VAE & GAN](#12-generative-models)
13. [Model Optimization & Mixed Precision](#13-optimization)
14. [Deployment — TFLite, TF Serving & FastAPI](#14-deployment)
15. [Capstone Projects](#15-capstone)


---
## 1. Environment Setup
### ⏱ Estimated Time: 30 minutes


### Installation

```bash
# Create a dedicated conda environment
conda create -n keras-dl python=3.10 -y
conda activate keras-dl

# Install core packages
pip install tensorflow==2.15.0
pip install keras==2.15.0
pip install numpy pandas matplotlib seaborn scikit-learn
pip install opencv-python pillow
pip install jupyter jupyterlab ipywidgets
pip install fastapi uvicorn

# GPU check (optional but recommended)
python -c "import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))"
```


In [ ]:
import tensorflow as tf
import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Pretty plots
plt.style.use('seaborn-v0_8-darkgrid')
matplotlib.rcParams['figure.dpi'] = 120

print(f'TensorFlow version : {tf.__version__}')
print(f'Keras version      : {keras.__version__}')
print(f'NumPy version      : {np.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available     : {len(gpus)}')
if gpus:
    for g in gpus:
        print(f'  → {g}')


---
## 2. Keras Core Concepts
### ⏱ Estimated Time: 2 hours

Keras organises deep learning into **three building blocks**:

| Block | Role | Example |
|---|---|---|
| **Layer** | Learnable computation unit | `Dense`, `Conv2D`, `LSTM` |
| **Model** | Container of layers | `Sequential`, `Model` |
| **Tensor** | Data that flows through | NumPy arrays or `tf.Tensor` |

### The Keras Training Loop in One Slide
```
Data ──▶ Model.forward() ──▶ Loss ──▶ Gradient ──▶ Optimizer.step() ──▶ repeat
```


In [ ]:
# Every Keras layer is callable on a tensor
dense = keras.layers.Dense(units=4, activation='relu')
x = tf.constant([[1.0, 2.0, 3.0]])  # shape (1, 3)
y = dense(x)                         # forward pass
print('Input shape  :', x.shape)
print('Output shape :', y.shape)
print('Output values:', y.numpy())
print('Weight shape :', dense.kernel.shape)   # (3, 4)
print('Bias shape   :', dense.bias.shape)     # (4,)


### Activation Functions — Visual Intuition

| Activation | Formula | Use Case |
|---|---|---|
| ReLU | `max(0, x)` | Hidden layers (default) |
| Sigmoid | `1/(1+e^-x)` | Binary classification output |
| Softmax | `e^xi / Σe^xj` | Multi-class output |
| Tanh | `(e^x - e^-x)/(e^x + e^-x)` | RNNs, normalised outputs |
| LeakyReLU | `max(αx, x)` | Avoid dying ReLU |


In [ ]:
x_vals = np.linspace(-4, 4, 300)
activations = {
    'ReLU':    tf.nn.relu,
    'Sigmoid': tf.nn.sigmoid,
    'Tanh':    tf.nn.tanh,
    'Softplus':tf.nn.softplus,
    'ELU':     tf.nn.elu,
}
fig, axes = plt.subplots(1, 5, figsize=(18, 3))
for ax, (name, fn) in zip(axes, activations.items()):
    y_vals = fn(x_vals).numpy()
    ax.plot(x_vals, y_vals, lw=2.5, color='steelblue')
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_ylim(-1.5, 2)
plt.suptitle('Activation Functions', fontsize=15, y=1.02)
plt.tight_layout(); plt.show()


---
## 3. Sequential API — Your First Neural Network
### ⏱ Estimated Time: 3 hours

The `Sequential` model is a **linear stack of layers** — output of one feeds directly into the next.  
Best for: simple feedforward networks, quick prototyping.

### 🎯 Project: Binary Classification on Synthetic Data


In [ ]:
# ── Generate data ───────────────────────────────────────────────────────────
X, y = make_classification(
    n_samples=5000, n_features=20, n_informative=10,
    n_redundant=5, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')


In [ ]:
# ── Build model ─────────────────────────────────────────────────────────────
tf.random.set_seed(42)
model = keras.Sequential([
    keras.layers.Input(shape=(20,)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid'),
], name='BinaryClassifier')

model.summary()


In [ ]:
# ── Compile & train ─────────────────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)
history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=50,
    batch_size=64,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, verbose=1)
    ],
    verbose=1
)


In [ ]:
# ── Plot training curves ─────────────────────────────────────────────────────
def plot_history(h):
    metrics = ['loss', 'accuracy', 'auc']
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, m in zip(axes, metrics):
        ax.plot(h.history[m],       label=f'Train {m}', lw=2)
        ax.plot(h.history[f'val_{m}'], label=f'Val {m}', lw=2, linestyle='--')
        ax.set_title(m.upper(), fontweight='bold')
        ax.legend(); ax.set_xlabel('Epoch')
    plt.tight_layout(); plt.show()

plot_history(history)
test_loss, test_acc, test_auc = model.evaluate(X_test, y_test, verbose=0)
print(f'\nTest  →  Loss: {test_loss:.4f}  Acc: {test_acc:.4f}  AUC: {test_auc:.4f}')


---
## 4. Functional API — Multi-Input & Multi-Output Models
### ⏱ Estimated Time: 3 hours

The Functional API builds models as **directed acyclic graphs (DAGs)**.  
Use it when you need: shared layers, multiple inputs/outputs, skip connections.

### 🎯 Project: Multi-Input Regression (Tabular + Embeddings)


In [ ]:
# ── Multi-input model: numeric + categorical ─────────────────────────────────
tf.random.set_seed(42)

# Inputs
numeric_input = keras.Input(shape=(10,),  name='numeric')
category_input= keras.Input(shape=(1,),   name='category')

# Numeric branch
x_num = keras.layers.Dense(64, activation='relu')(numeric_input)
x_num = keras.layers.BatchNormalization()(x_num)
x_num = keras.layers.Dense(32, activation='relu')(x_num)

# Category branch — embedding
x_cat = keras.layers.Embedding(input_dim=10, output_dim=8)(category_input)
x_cat = keras.layers.Flatten()(x_cat)
x_cat = keras.layers.Dense(16, activation='relu')(x_cat)

# Merge
merged = keras.layers.Concatenate()([x_num, x_cat])
merged = keras.layers.Dense(64, activation='relu')(merged)
merged = keras.layers.Dropout(0.3)(merged)

# Two outputs: main regression + auxiliary classification
main_output = keras.layers.Dense(1, name='regression')(merged)
aux_output  = keras.layers.Dense(1, activation='sigmoid', name='binary')(merged)

multi_model = keras.Model(
    inputs=[numeric_input, category_input],
    outputs=[main_output, aux_output],
    name='MultiIO_Model'
)
multi_model.summary()


In [ ]:
# ── Compile with per-output loss weights ─────────────────────────────────────
multi_model.compile(
    optimizer='adam',
    loss={'regression': 'mse', 'binary': 'binary_crossentropy'},
    loss_weights={'regression': 1.0, 'binary': 0.3},
    metrics={'regression': 'mae', 'binary': 'accuracy'}
)

# Synthetic dummy data
n = 2000
X_num  = np.random.randn(n, 10).astype('float32')
X_cat  = np.random.randint(0, 10, (n, 1)).astype('int32')
y_reg  = (X_num[:, 0] * 3 + np.random.randn(n) * 0.5).astype('float32')
y_bin  = (y_reg > 0).astype('float32')

multi_model.fit(
    {'numeric': X_num, 'category': X_cat},
    {'regression': y_reg, 'binary': y_bin},
    epochs=30, batch_size=64, validation_split=0.2, verbose=0
)
print('Multi-output model training complete ✓')


---
## 5. Model Subclassing — Full Custom Control
### ⏱ Estimated Time: 3 hours

Subclassing `keras.Model` gives you **maximum flexibility**:  
→ Custom forward pass logic, dynamic computation, research models.


In [ ]:
class ResidualBlock(keras.layers.Layer):
    """Pre-activation Residual Block."""
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.bn1   = keras.layers.BatchNormalization()
        self.dense1= keras.layers.Dense(units, activation='relu')
        self.bn2   = keras.layers.BatchNormalization()
        self.dense2= keras.layers.Dense(units)
        self.proj  = keras.layers.Dense(units)  # projection shortcut

    def call(self, x, training=False):
        shortcut = self.proj(x)
        x = self.bn1(x, training=training)
        x = self.dense1(x)
        x = self.bn2(x, training=training)
        x = self.dense2(x)
        return keras.activations.relu(x + shortcut)


class DeepResNet(keras.Model):
    """Stack of residual blocks for tabular data."""
    def __init__(self, num_blocks=4, units=128, num_classes=10, **kwargs):
        super().__init__(**kwargs)
        self.stem   = keras.layers.Dense(units, activation='relu')
        self.blocks = [ResidualBlock(units, name=f'res_{i}') for i in range(num_blocks)]
        self.head   = keras.layers.Dense(num_classes, activation='softmax')

    def call(self, x, training=False):
        x = self.stem(x)
        for block in self.blocks:
            x = block(x, training=training)
        return self.head(x)


# Quick smoke-test
resnet = DeepResNet(num_blocks=3, units=64, num_classes=5)
dummy = tf.random.normal([8, 20])
out = resnet(dummy, training=True)
print('Output shape:', out.shape)  # (8, 5)
resnet.summary()


---
## 6. Custom Layers, Losses & Metrics
### ⏱ Estimated Time: 3 hours


In [ ]:
# ── Custom Layer: Scaled Dot-Product Attention ───────────────────────────────
class ScaledDotAttention(keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, query, key, value, mask=None):
        d_k  = tf.cast(tf.shape(key)[-1], tf.float32)
        scores= tf.matmul(query, key, transpose_b=True) / tf.sqrt(d_k)
        if mask is not None:
            scores += mask * -1e9
        weights = tf.nn.softmax(scores, axis=-1)
        return tf.matmul(weights, value), weights


# ── Custom Loss: Focal Loss (for class imbalance) ─────────────────────────────
class FocalLoss(keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        bce    = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        p_t    = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        focal  = self.alpha * tf.pow(1 - p_t, self.gamma) * bce
        return tf.reduce_mean(focal)


# ── Custom Metric: F1 Score ───────────────────────────────────────────────────
class F1Score(keras.metrics.Metric):
    def __init__(self, threshold=0.5, **kwargs):
        super().__init__(**kwargs)
        self.threshold = threshold
        self.tp = self.add_weight('tp', initializer='zeros')
        self.fp = self.add_weight('fp', initializer='zeros')
        self.fn = self.add_weight('fn', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred_bin = tf.cast(y_pred >= self.threshold, tf.float32)
        y_true     = tf.cast(y_true, tf.float32)
        self.tp.assign_add(tf.reduce_sum(y_true * y_pred_bin))
        self.fp.assign_add(tf.reduce_sum((1 - y_true) * y_pred_bin))
        self.fn.assign_add(tf.reduce_sum(y_true * (1 - y_pred_bin)))

    def result(self):
        precision = self.tp / (self.tp + self.fp + 1e-7)
        recall    = self.tp / (self.tp + self.fn + 1e-7)
        return 2 * precision * recall / (precision + recall + 1e-7)

    def reset_state(self):
        for v in self.variables:
            v.assign(tf.zeros_like(v))


print('Custom components defined:')
print('  ✓ ScaledDotAttention layer')
print('  ✓ FocalLoss (γ=2, α=0.25)')
print('  ✓ F1Score metric')
# Verify focal loss
fl = FocalLoss()
y_t = tf.constant([1.0, 0.0, 1.0, 1.0])
y_p = tf.constant([0.9, 0.1, 0.4, 0.8])
print(f'\nFocal loss sample: {fl(y_t, y_p).numpy():.4f}')


---
## 7. Callbacks & Training Control
### ⏱ Estimated Time: 2 hours

Callbacks are **hooks** that fire at training events: epoch start/end, batch start/end, training start/end.

| Callback | Purpose |
|---|---|
| `EarlyStopping` | Stop when val loss stops improving |
| `ModelCheckpoint` | Save best weights automatically |
| `ReduceLROnPlateau` | Lower LR when stuck |
| `TensorBoard` | Real-time training dashboard |
| Custom | Any logic you can write in Python |


In [ ]:
class WarmUpCosineDecay(keras.callbacks.Callback):
    """Linear warm-up then cosine annealing learning rate schedule."""
    def __init__(self, warmup_epochs, total_epochs, base_lr, min_lr=1e-6):
        super().__init__()
        self.warmup_epochs  = warmup_epochs
        self.total_epochs   = total_epochs
        self.base_lr        = base_lr
        self.min_lr         = min_lr
        self.lr_history      = []

    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            lr = self.base_lr * (epoch + 1) / self.warmup_epochs
        else:
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + np.cos(np.pi * progress))
        self.model.optimizer.learning_rate.assign(lr)
        self.lr_history.append(float(lr))


class ConfusionMatrixCallback(keras.callbacks.Callback):
    """Log confusion matrix at end of each epoch."""
    def __init__(self, val_data, class_names):
        super().__init__()
        self.X_val, self.y_val = val_data
        self.class_names = class_names

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 10 == 0:
            y_pred = np.argmax(self.model.predict(self.X_val, verbose=0), axis=1)
            from sklearn.metrics import confusion_matrix
            cm = confusion_matrix(self.y_val, y_pred)
            print(f'\nEpoch {epoch+1} — Confusion Matrix:\n{cm}')


# Demo: plot the warm-up cosine LR schedule
sched = WarmUpCosineDecay(warmup_epochs=5, total_epochs=60, base_lr=1e-3)
# Simulate calling on_epoch_begin
class FakeModel:
    class optimizer:
        class learning_rate:
            val = 0.0
            @staticmethod
            def assign(v): FakeModel.optimizer.learning_rate.val = v
    model = None
sched.model = FakeModel()
for e in range(60):
    sched.on_epoch_begin(e)
plt.figure(figsize=(10, 3))
plt.plot(sched.lr_history, lw=2.5, color='coral')
plt.axvline(5, ls='--', color='gray', label='End of warm-up')
plt.title('Warm-Up + Cosine Annealing LR Schedule', fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Learning Rate')
plt.legend(); plt.tight_layout(); plt.show()


---
## 8. Data Pipelines with tf.data
### ⏱ Estimated Time: 3 hours

`tf.data` builds **fast, scalable, GPU-prefetched** input pipelines.  
**Golden rule:** the GPU should never wait for the CPU to prepare data.


In [ ]:
import os, pathlib

# ── Optimal tf.data pipeline ─────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

def build_pipeline(X, y, batch_size=64, augment=False, shuffle=True):
    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(X), seed=42)

    def augment_fn(x, label):
        # For image data: random flip, brightness, contrast
        x = tf.image.random_flip_left_right(x) if augment else x
        return x, label

    dataset = dataset.map(augment_fn,      num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(batch_size,    drop_remainder=True)
    dataset = dataset.prefetch(AUTOTUNE)   # overlap data prep & GPU compute
    dataset = dataset.cache()              # cache in RAM after 1st epoch
    return dataset


# Demonstrate with synthetic tabular data
X_demo = np.random.randn(10000, 20).astype('float32')
y_demo = np.random.randint(0, 2, 10000).astype('float32')
train_ds = build_pipeline(X_demo, y_demo, batch_size=128)

# Inspect one batch
for batch_x, batch_y in train_ds.take(1):
    print('Batch X shape:', batch_x.shape)
    print('Batch y shape:', batch_y.shape)
print(f'\nTotal batches per epoch: {len(train_ds)}')


In [ ]:
# ── Image pipeline from directory (CIFAR-10 style) ───────────────────────────
IMG_SIZE = 32
NUM_CLASSES = 10

def preprocess_image(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    return image, label

def augment_image(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.15)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

# Load CIFAR-10 via Keras
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.flatten(); y_test = y_test.flatten()
print(f'CIFAR-10 — Train: {x_train.shape}  Test: {x_test.shape}')

train_dataset = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(50000)
    .map(preprocess_image, num_parallel_calls=AUTOTUNE)
    .map(augment_image,    num_parallel_calls=AUTOTUNE)
    .batch(128)
    .prefetch(AUTOTUNE)
)
test_dataset = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(preprocess_image, num_parallel_calls=AUTOTUNE)
    .batch(128)
    .prefetch(AUTOTUNE)
)
# Visualise augmented samples
sample_imgs, sample_labels = next(iter(train_dataset))
cifar_names = ['airplane','automobile','bird','cat','deer',
                'dog','frog','horse','ship','truck']
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(sample_imgs[i].numpy())
    ax.set_title(cifar_names[sample_labels[i]], fontsize=8)
    ax.axis('off')
plt.suptitle('CIFAR-10 — Augmented Training Samples', fontweight='bold')
plt.tight_layout(); plt.show()


---
## 9. Computer Vision — CNNs & Transfer Learning
### ⏱ Estimated Time: 8 hours

### What a CNN Actually Does
```
Input Image  →  [Conv2D → BN → ReLU → Pool] × N  →  Flatten  →  Dense  →  Output
                 ← feature extraction →                ← classification →
```

| Layer | Role |
|---|---|
| `Conv2D(32, 3×3)` | 32 learnable filters detect edges, textures |
| `BatchNorm` | Normalize activations → faster, stabler training |
| `MaxPool2D(2×2)` | Downsample by 2×, add spatial invariance |
| `GlobalAvgPool2D` | Collapse spatial dims → feature vector |

### 🎯 Project A: Custom CNN on CIFAR-10


In [ ]:
def build_cnn(num_classes=10):
    return keras.Sequential([
        keras.layers.Input((32, 32, 3)),
        # Block 1
        keras.layers.Conv2D(32, 3, padding='same'), keras.layers.BatchNormalization(),
        keras.layers.Activation('relu'),
        keras.layers.Conv2D(32, 3, padding='same'), keras.layers.BatchNormalization(),
        keras.layers.Activation('relu'),
        keras.layers.MaxPooling2D(2), keras.layers.Dropout(0.2),
        # Block 2
        keras.layers.Conv2D(64, 3, padding='same'), keras.layers.BatchNormalization(),
        keras.layers.Activation('relu'),
        keras.layers.Conv2D(64, 3, padding='same'), keras.layers.BatchNormalization(),
        keras.layers.Activation('relu'),
        keras.layers.MaxPooling2D(2), keras.layers.Dropout(0.3),
        # Block 3
        keras.layers.Conv2D(128, 3, padding='same'), keras.layers.BatchNormalization(),
        keras.layers.Activation('relu'),
        keras.layers.GlobalAveragePooling2D(),
        # Head
        keras.layers.Dense(256, activation='relu'),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(num_classes, activation='softmax')
    ], name='CustomCNN')

cnn = build_cnn()
cnn.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
cnn.summary()


In [ ]:
# ── Train ────────────────────────────────────────────────────────────────────
cnn_history = cnn.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=40,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
    ]
)
plot_history(cnn_history)
print('\nTest evaluation:')
cnn.evaluate(test_dataset)


### 🎯 Project B: Transfer Learning with EfficientNetV2


In [ ]:
# Fine-tune EfficientNetV2-S on CIFAR-10
# Phase 1: Feature extraction (frozen backbone)
base = keras.applications.EfficientNetV2S(
    include_top=False,
    weights='imagenet',
    input_shape=(96, 96, 3)   # upsampled from 32×32
)
base.trainable = False

inputs  = keras.Input((32, 32, 3))
x = keras.layers.Resizing(96, 96)(inputs)          # upscale
x = keras.applications.efficientnet_v2.preprocess_input(x)
x = base(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.4)(x)
outputs = keras.layers.Dense(10, activation='softmax')(x)
effnet_model = keras.Model(inputs, outputs, name='EfficientNetV2_TL')

effnet_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print('Phase 1: Training classification head...')
effnet_model.fit(train_dataset, validation_data=test_dataset,
                 epochs=5, verbose=1)

# Phase 2: Fine-tune top layers
base.trainable = True
# Freeze everything except the last 30 layers
for layer in base.layers[:-30]:
    layer.trainable = False

effnet_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),   # much lower LR
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print('\nPhase 2: Fine-tuning top 30 backbone layers...')
effnet_model.fit(train_dataset, validation_data=test_dataset,
                 epochs=10, verbose=1)


In [ ]:
# ── Grad-CAM Visualisation ───────────────────────────────────────────────────
def grad_cam(model, image, layer_name, class_idx=None):
    """Compute Grad-CAM heatmap for a single image."""
    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(layer_name).output, model.output]
    )
    img_batch = tf.expand_dims(image, 0)
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_batch)
        if class_idx is None:
            class_idx = tf.argmax(preds[0])
        loss = preds[:, class_idx]
    grads   = tape.gradient(loss, conv_out)
    pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam     = tf.reduce_sum(conv_out[0] * pooled, axis=-1).numpy()
    cam     = np.maximum(cam, 0)
    cam    /= (cam.max() + 1e-8)
    return cam

print('Grad-CAM utility defined.')
print('Usage: heatmap = grad_cam(model, image, last_conv_layer_name)')


---
## 10. Natural Language Processing — Text & Transformers
### ⏱ Estimated Time: 8 hours

### 🎯 Project A: Sentiment Analysis (IMDB)


In [ ]:
# ── Load & preprocess IMDB ───────────────────────────────────────────────────
VOCAB_SIZE  = 20000
MAX_LEN     = 256
EMBED_DIM   = 128

(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = keras.datasets.imdb.load_data(
    num_words=VOCAB_SIZE
)
x_train_pad = keras.preprocessing.sequence.pad_sequences(
    x_train_raw, maxlen=MAX_LEN, padding='post', truncating='post'
)
x_test_pad = keras.preprocessing.sequence.pad_sequences(
    x_test_raw,  maxlen=MAX_LEN, padding='post', truncating='post'
)
print(f'Train: {x_train_pad.shape}  Test: {x_test_pad.shape}')


In [ ]:
# ── TextCNN model ────────────────────────────────────────────────────────────
def build_text_cnn(vocab_size, max_len, embed_dim, num_filters=128):
    inp = keras.Input(shape=(max_len,))
    x   = keras.layers.Embedding(vocab_size, embed_dim)(inp)

    # Parallel convolutions with different kernel sizes
    pools = []
    for k in [2, 3, 4, 5]:
        conv = keras.layers.Conv1D(num_filters, k, activation='relu', padding='valid')(x)
        pool = keras.layers.GlobalMaxPooling1D()(conv)
        pools.append(pool)

    concat = keras.layers.Concatenate()(pools)
    drop   = keras.layers.Dropout(0.5)(concat)
    output = keras.layers.Dense(1, activation='sigmoid')(drop)
    return keras.Model(inp, output, name='TextCNN')

text_cnn = build_text_cnn(VOCAB_SIZE, MAX_LEN, EMBED_DIM)
text_cnn.compile(optimizer='adam', loss='binary_crossentropy',
                 metrics=['accuracy', keras.metrics.AUC(name='auc')])
text_cnn.summary()


In [ ]:
# ── Train TextCNN ────────────────────────────────────────────────────────────
tcnn_hist = text_cnn.fit(
    x_train_pad, y_train_raw,
    validation_split=0.1,
    epochs=10, batch_size=128,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
)
loss, acc, auc = text_cnn.evaluate(x_test_pad, y_test_raw, verbose=0)
print(f'IMDB Test → Acc: {acc:.4f}  AUC: {auc:.4f}')


### 🎯 Project B: Transformer Encoder from Scratch


In [ ]:
class MultiHeadSelfAttention(keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.embed_dim = embed_dim
        self.Wq = keras.layers.Dense(embed_dim)
        self.Wk = keras.layers.Dense(embed_dim)
        self.Wv = keras.layers.Dense(embed_dim)
        self.Wo = keras.layers.Dense(embed_dim)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.head_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, x, training=False):
        B = tf.shape(x)[0]
        Q = self.split_heads(self.Wq(x), B)
        K = self.split_heads(self.Wk(x), B)
        V = self.split_heads(self.Wv(x), B)
        scale  = tf.sqrt(tf.cast(self.head_dim, tf.float32))
        scores = tf.matmul(Q, K, transpose_b=True) / scale
        weights= tf.nn.softmax(scores, axis=-1)
        ctx    = tf.matmul(weights, V)
        ctx    = tf.transpose(ctx, perm=[0, 2, 1, 3])
        ctx    = tf.reshape(ctx, (B, -1, self.embed_dim))
        return self.Wo(ctx)


class TransformerBlock(keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attn  = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ff    = keras.Sequential([
            keras.layers.Dense(ff_dim, activation='gelu'),
            keras.layers.Dense(embed_dim)
        ])
        self.ln1   = keras.layers.LayerNormalization()
        self.ln2   = keras.layers.LayerNormalization()
        self.drop1 = keras.layers.Dropout(dropout_rate)
        self.drop2 = keras.layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        # Pre-LN (more stable than post-LN)
        x = x + self.drop1(self.attn(self.ln1(x), training=training), training=training)
        x = x + self.drop2(self.ff(self.ln2(x)), training=training)
        return x


# Build transformer sentiment classifier
inp    = keras.Input(shape=(MAX_LEN,))
embed  = keras.layers.Embedding(VOCAB_SIZE, 64)(inp)
# Positional encoding via learnable embedding
pos    = tf.range(start=0, limit=MAX_LEN, delta=1)
pos_embed = keras.layers.Embedding(MAX_LEN, 64)(pos)
x      = embed + pos_embed
x      = TransformerBlock(64, 4, 256, 0.1)(x)
x      = TransformerBlock(64, 4, 256, 0.1)(x)
x      = keras.layers.GlobalAveragePooling1D()(x)
x      = keras.layers.Dropout(0.3)(x)
x      = keras.layers.Dense(64, activation='relu')(x)
out    = keras.layers.Dense(1, activation='sigmoid')(x)
transformer_clf = keras.Model(inp, out, name='TransformerClassifier')
transformer_clf.compile(optimizer=keras.optimizers.Adam(2e-4),
                         loss='binary_crossentropy',
                         metrics=['accuracy'])
transformer_clf.summary()


In [ ]:
t_hist = transformer_clf.fit(
    x_train_pad, y_train_raw,
    validation_split=0.1,
    epochs=8, batch_size=64,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
)
loss, acc = transformer_clf.evaluate(x_test_pad, y_test_raw, verbose=0)
print(f'Transformer Test → Acc: {acc:.4f}')


---
## 11. Time Series Forecasting
### ⏱ Estimated Time: 4 hours

### 🎯 Project: Stacked LSTM + CNN-LSTM Hybrid for Multivariate Forecasting


In [ ]:
# ── Generate synthetic multivariate time series ──────────────────────────────
def make_time_series(n=3000, n_features=5):
    t = np.linspace(0, 60*np.pi, n)
    X = np.column_stack([
        np.sin(t) + 0.1*np.random.randn(n),
        np.cos(t/2) + 0.1*np.random.randn(n),
        np.sin(t/3 + 1) + 0.05*np.random.randn(n),
        np.random.randn(n) * 0.5,
        np.cumsum(np.random.randn(n) * 0.01)
    ])
    # Target: next step of first feature
    y = np.roll(X[:, 0], -1)
    return X[:-1], y[:-1]

def make_windows(X, y, window=60, horizon=1):
    xs, ys = [], []
    for i in range(len(X) - window - horizon + 1):
        xs.append(X[i:i+window])
        ys.append(y[i+window:i+window+horizon])
    return np.array(xs, dtype='float32'), np.array(ys, dtype='float32').squeeze()

X_ts, y_ts = make_time_series(3000)
X_ts = (X_ts - X_ts.mean(0)) / (X_ts.std(0) + 1e-8)
Xw, yw = make_windows(X_ts, y_ts, window=60)
split = int(0.8 * len(Xw))
Xw_tr, Xw_te = Xw[:split], Xw[split:]
yw_tr, yw_te = yw[:split], yw[split:]
print(f'Window shape: {Xw.shape}  Target: {yw.shape}')


In [ ]:
# ── CNN-LSTM Hybrid ──────────────────────────────────────────────────────────
def build_cnn_lstm(window, n_features):
    inp = keras.Input((window, n_features))
    # CNN feature extraction
    x = keras.layers.Conv1D(64, 3, padding='same', activation='relu')(inp)
    x = keras.layers.Conv1D(64, 3, padding='same', activation='relu')(x)
    x = keras.layers.MaxPooling1D(2)(x)
    # Bidirectional LSTM sequence modelling
    x = keras.layers.Bidirectional(keras.layers.LSTM(128, return_sequences=True))(x)
    x = keras.layers.Bidirectional(keras.layers.LSTM(64))(x)
    # Attention-like weighting
    x = keras.layers.Dense(64, activation='relu')(x)
    x = keras.layers.Dropout(0.3)(x)
    out = keras.layers.Dense(1)(x)
    model = keras.Model(inp, out, name='CNN_BiLSTM')
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='mse', metrics=['mae'])
    return model

ts_model = build_cnn_lstm(60, 5)
ts_model.summary()
ts_hist = ts_model.fit(
    Xw_tr, yw_tr,
    validation_data=(Xw_te, yw_te),
    epochs=30, batch_size=64,
    callbacks=[keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True)]
)
preds = ts_model.predict(Xw_te).flatten()
plt.figure(figsize=(14, 4))
plt.plot(yw_te[:300], label='True', lw=1.5)
plt.plot(preds[:300], label='Predicted', lw=1.5, linestyle='--')
plt.title('CNN-BiLSTM Time Series Forecast', fontweight='bold')
plt.legend(); plt.tight_layout(); plt.show()


---
## 12. Generative Models — VAE & GAN
### ⏱ Estimated Time: 8 hours

### 🎯 Project A: Variational Autoencoder on Fashion-MNIST


In [ ]:
# ── VAE Sampling Layer ───────────────────────────────────────────────────────
class Sampling(keras.layers.Layer):
    """Reparameterisation trick: z = mu + eps * sigma."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        eps = tf.random.normal(tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * eps


LATENT_DIM = 16
(fmnist_x, _), _ = keras.datasets.fashion_mnist.load_data()
fmnist_x = fmnist_x.astype('float32') / 255.0
fmnist_x = fmnist_x[..., np.newaxis]  # (60000, 28, 28, 1)

# Encoder
enc_inp = keras.Input((28, 28, 1))
x_e = keras.layers.Conv2D(32, 3, strides=2, padding='same', activation='relu')(enc_inp)
x_e = keras.layers.Conv2D(64, 3, strides=2, padding='same', activation='relu')(x_e)
x_e = keras.layers.Flatten()(x_e)
x_e = keras.layers.Dense(256, activation='relu')(x_e)
z_mean    = keras.layers.Dense(LATENT_DIM, name='z_mean')(x_e)
z_log_var = keras.layers.Dense(LATENT_DIM, name='z_log_var')(x_e)
z         = Sampling()([z_mean, z_log_var])
encoder   = keras.Model(enc_inp, [z_mean, z_log_var, z], name='encoder')

# Decoder
dec_inp = keras.Input((LATENT_DIM,))
x_d = keras.layers.Dense(7 * 7 * 64, activation='relu')(dec_inp)
x_d = keras.layers.Reshape((7, 7, 64))(x_d)
x_d = keras.layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu')(x_d)
x_d = keras.layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x_d)
dec_out = keras.layers.Conv2DTranspose(1, 3, padding='same', activation='sigmoid')(x_d)
decoder = keras.Model(dec_inp, dec_out, name='decoder')
print('Encoder & Decoder built.')
encoder.summary()


In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, beta=1.0, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.beta    = beta   # β-VAE: β>1 encourages disentanglement
        self.recon_tracker = keras.metrics.Mean(name='recon_loss')
        self.kl_tracker    = keras.metrics.Mean(name='kl_loss')
        self.total_tracker = keras.metrics.Mean(name='total_loss')

    @property
    def metrics(self):
        return [self.total_tracker, self.recon_tracker, self.kl_tracker]

    def train_step(self, data):
        x = data[0] if isinstance(data, tuple) else data
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(x, training=True)
            x_recon = self.decoder(z, training=True)
            recon_loss = tf.reduce_mean(
                tf.reduce_sum(keras.losses.binary_crossentropy(x, x_recon), axis=(1, 2))
            )
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1)
            )
            total_loss = recon_loss + self.beta * kl_loss
        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self.recon_tracker.update_state(recon_loss)
        self.kl_tracker.update_state(kl_loss)
        self.total_tracker.update_state(total_loss)
        return {m.name: m.result() for m in self.metrics}

vae = VAE(encoder, decoder, beta=1.0)
vae.compile(optimizer=keras.optimizers.Adam(1e-3))
vae_hist = vae.fit(fmnist_x, epochs=20, batch_size=128, verbose=1)


In [ ]:
# ── Sample from latent space ─────────────────────────────────────────────────
n_samples = 16
z_sample = np.random.randn(n_samples, LATENT_DIM).astype('float32')
generated = decoder.predict(z_sample)
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
plt.suptitle('VAE Generated Fashion-MNIST Samples', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()


### 🎯 Project B: DCGAN


In [ ]:
# ── Generator & Discriminator ────────────────────────────────────────────────
NOISE_DIM = 128

def build_generator():
    return keras.Sequential([
        keras.layers.Input((NOISE_DIM,)),
        keras.layers.Dense(7 * 7 * 256),
        keras.layers.Reshape((7, 7, 256)),
        keras.layers.Conv2DTranspose(128, 4, strides=2, padding='same'),
        keras.layers.BatchNormalization(), keras.layers.LeakyReLU(0.2),
        keras.layers.Conv2DTranspose(64,  4, strides=2, padding='same'),
        keras.layers.BatchNormalization(), keras.layers.LeakyReLU(0.2),
        keras.layers.Conv2DTranspose(1,   3, padding='same', activation='tanh'),
    ], name='Generator')

def build_discriminator():
    return keras.Sequential([
        keras.layers.Input((28, 28, 1)),
        keras.layers.Conv2D(64,  4, strides=2, padding='same'), keras.layers.LeakyReLU(0.2),
        keras.layers.Dropout(0.3),
        keras.layers.Conv2D(128, 4, strides=2, padding='same'), keras.layers.LeakyReLU(0.2),
        keras.layers.Dropout(0.3),
        keras.layers.Flatten(),
        keras.layers.Dense(1, activation='sigmoid')
    ], name='Discriminator')

class DCGAN(keras.Model):
    def __init__(self, generator, discriminator, **kwargs):
        super().__init__(**kwargs)
        self.G = generator
        self.D = discriminator
        self.cross_entropy = keras.losses.BinaryCrossentropy(label_smoothing=0.1)
        self.g_loss_tracker = keras.metrics.Mean('g_loss')
        self.d_loss_tracker = keras.metrics.Mean('d_loss')

    def compile(self, g_opt, d_opt):
        super().compile()
        self.g_opt = g_opt
        self.d_opt = d_opt

    def train_step(self, real_images):
        batch_size = tf.shape(real_images)[0]
        noise = tf.random.normal([batch_size, NOISE_DIM])
        # Train discriminator
        with tf.GradientTape() as dt:
            fake = self.G(noise, training=True)
            real_logits = self.D(real_images, training=True)
            fake_logits = self.D(fake, training=True)
            d_loss = (
                self.cross_entropy(tf.ones_like(real_logits),  real_logits) +
                self.cross_entropy(tf.zeros_like(fake_logits), fake_logits)
            )
        self.d_opt.apply_gradients(zip(dt.gradient(d_loss, self.D.trainable_variables),
                                       self.D.trainable_variables))
        # Train generator
        with tf.GradientTape() as gt:
            fake     = self.G(noise, training=True)
            logits   = self.D(fake, training=False)
            g_loss   = self.cross_entropy(tf.ones_like(logits), logits)
        self.g_opt.apply_gradients(zip(gt.gradient(g_loss, self.G.trainable_variables),
                                       self.G.trainable_variables))
        self.g_loss_tracker.update_state(g_loss)
        self.d_loss_tracker.update_state(d_loss)
        return {'g_loss': self.g_loss_tracker.result(),
                'd_loss': self.d_loss_tracker.result()}

G = build_generator()
D = build_discriminator()
dcgan = DCGAN(G, D)
dcgan.compile(
    g_opt=keras.optimizers.Adam(2e-4, beta_1=0.5),
    d_opt=keras.optimizers.Adam(2e-4, beta_1=0.5)
)
# Normalize to [-1, 1] for tanh output
gan_data = (fmnist_x * 2) - 1
gan_ds   = tf.data.Dataset.from_tensor_slices(gan_data).shuffle(60000).batch(128).prefetch(AUTOTUNE)
dcgan.fit(gan_ds, epochs=30, verbose=1)


---
## 13. Model Optimization & Mixed Precision
### ⏱ Estimated Time: 3 hours


In [ ]:
# ── Mixed Precision Training (2× speed on modern GPUs) ───────────────────────
from tensorflow.keras import mixed_precision

# Enable globally
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print('Compute dtype   :', policy.compute_dtype)   # float16
print('Variable dtype  :', policy.variable_dtype)  # float32

# Build model (same API — fp16 handled automatically)
mp_model = keras.Sequential([
    keras.layers.Input((20,)),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid', dtype='float32')  # output must be float32
])
# LossScaleOptimizer wraps optimizer to prevent fp16 underflow
optimizer = keras.optimizers.Adam(1e-3)
# (In TF2.x, mixed_precision.LossScaleOptimizer is applied automatically)
mp_model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
print('Mixed precision model compiled ✓')

# Restore default policy
mixed_precision.set_global_policy('float32')


In [ ]:
# ── Quantization-Aware Training (QAT) ────────────────────────────────────────
# Install: pip install tensorflow-model-optimization
try:
    import tensorflow_model_optimization as tfmot
    quantize_model = tfmot.quantization.keras.quantize_model

    base_model = keras.Sequential([
        keras.layers.Input((20,)),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    base_model.compile(optimizer='adam', loss='binary_crossentropy')

    q_aware_model = quantize_model(base_model)
    q_aware_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    print('QAT model built ✓')
    q_aware_model.summary()
except ImportError:
    print('tensorflow-model-optimization not installed.')
    print('Run: pip install tensorflow-model-optimization')


---
## 14. Deployment — SavedModel, TFLite & FastAPI
### ⏱ Estimated Time: 4 hours


In [ ]:
import os, tempfile, json

# ── Save & load as SavedModel ─────────────────────────────────────────────────
SAVE_PATH = '/tmp/keras_saved_model'
cnn.save(SAVE_PATH)
print(f'Model saved to: {SAVE_PATH}')

loaded_model = keras.models.load_model(SAVE_PATH)
sample_batch, _ = next(iter(test_dataset))
preds_orig   = cnn.predict(sample_batch[:4], verbose=0)
preds_loaded = loaded_model.predict(sample_batch[:4], verbose=0)
print('Max output diff (should be ~0):', np.abs(preds_orig - preds_loaded).max())


In [ ]:
# ── Convert to TFLite (mobile deployment) ────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_saved_model(SAVE_PATH)
converter.optimizations = [tf.lite.Optimize.DEFAULT]   # post-training quantisation
tflite_model = converter.convert()
TFLITE_PATH = '/tmp/cnn_model.tflite'
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)
orig_mb   = os.path.getsize(SAVE_PATH + '/saved_model.pb') / 1e6
tflite_mb = os.path.getsize(TFLITE_PATH) / 1e6
print(f'Original SavedModel:   {orig_mb:.2f} MB')
print(f'TFLite (quantized):    {tflite_mb:.2f} MB')
print(f'Compression ratio:     {orig_mb/tflite_mb:.1f}×')


In [ ]:
# ── FastAPI REST endpoint template ────────────────────────────────────────────
fastapi_code = '''
# app.py  — Run with: uvicorn app:app --host 0.0.0.0 --port 8000
import numpy as np
import keras
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List

app   = FastAPI(title='CIFAR-10 Classifier API', version='1.0')
model = keras.models.load_model('/tmp/keras_saved_model')
CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']

class ImageInput(BaseModel):
    pixels: List[List[List[List[float]]]]  # [batch, H, W, C]

class PredictionOutput(BaseModel):
    predictions: List[str]
    confidences: List[float]

@app.get('/')
def health(): return {'status': 'healthy'}

@app.post('/predict', response_model=PredictionOutput)
def predict(data: ImageInput):
    x    = np.array(data.pixels, dtype='float32') / 255.0
    pred = model.predict(x)
    idxs = pred.argmax(axis=1)
    return PredictionOutput(
        predictions=[CLASSES[i] for i in idxs],
        confidences=[float(pred[j, i]) for j, i in enumerate(idxs)]
    )
'''
print('FastAPI app template:')
print(fastapi_code)
# Save for reference
with open('/tmp/keras_fastapi_app.py', 'w') as f:
    f.write(fastapi_code.strip())
print('\nSaved to /tmp/keras_fastapi_app.py')


---
## 15. Capstone Projects
### ⏱ Estimated Time: 20–30 hours

### 🏆 Capstone 1: End-to-End Image Classification System
```
Raw Images → Data Pipeline → Transfer Learning (EfficientNetV2) → Fine-tuning
           → Evaluation (Confusion Matrix, ROC) → TFLite Export → FastAPI → Docker
```

### 🏆 Capstone 2: Transformer NLP Pipeline
```
Raw Text → Tokenisation → Custom Transformer → Multi-label Classification
         → SHAP Explainability → REST API → CI/CD Pipeline
```

### 🏆 Capstone 3: Generative AI App
```
Train DCGAN on custom dataset → FID evaluation → Latent space interpolation
                              → Interactive Streamlit demo → Deploy to HuggingFace Spaces
```

---

### 📋 Suggested GitHub Project Structure
```
deep-learning-keras/
├── data/              # raw, processed, external
├── notebooks/         # this file and experiments
├── src/
│   ├── data.py        # tf.data pipelines
│   ├── models.py      # model definitions
│   ├── train.py       # training scripts
│   ├── evaluate.py    # metrics, visualisations
│   └── predict.py     # inference utilities
├── api/
│   ├── app.py         # FastAPI application
│   └── Dockerfile
├── configs/           # YAML hyperparameter files
├── tests/             # unit tests
├── requirements.txt
└── README.md
```

---

## 🗓 6-Month Keras Mastery Plan

| Month | Focus | Deliverable |
|---|---|---|
| 1 | Math + Python + Sequential API | Binary classifier on real dataset |
| 2 | Functional API + Custom components | Multi-input model with custom loss |
| 3 | CNNs + Transfer Learning | CIFAR-10 / ImageNet top-5 accuracy |
| 4 | NLP + Transformers | Sentiment classifier >93% accuracy |
| 5 | Generative Models + Time Series | VAE / GAN generating realistic images |
| 6 | Deployment + Capstone | Dockerised FastAPI + TFLite mobile app |

---
*This notebook is part of the Complete Deep Learning Bootcamp Curriculum*  
*© Premium Deep Learning Institute — All rights reserved*
